# 02 — Preprocessing & Feature Engineering

Phase ML-3 (first half). Builds on `01_dataset_audit.ipynb`. Produces the
cleaned, harmonised, adult-only dataset and resolves the Height/Weight/BMI
representation question empirically (per the user's instruction: do not
assume all three are independent features — determine this via experiment).

Outputs `train.pkl` / `test.pkl` (stratified 80/20 split, fixed seed) used by
the next two notebooks — the 20% test set is set aside here and not touched
again until final model evaluation.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
TARGET = 'Heart Disease'
pd.set_option('display.width', 160)

## 1. Load and clean (column names, text-artifact parsing)

In [2]:
DATA_PATH = "../Resource files/Heart_diasease_dataset_from_Northern_Bangladesh.xlsx"
df = pd.read_excel(DATA_PATH, sheet_name="Our Dataset")
df.columns = [c.strip() for c in df.columns]
print("Loaded shape:", df.shape)

def clean_numeric_text(series):
    def parse(v):
        if pd.isna(v):
            return np.nan
        s = str(v).strip().replace('`', '').replace(' ', '').replace(',', '.')
        try:
            return float(s)
        except ValueError:
            return np.nan
    return series.apply(parse)

for col in ['Himoglobin', 'Potassium', 'Chloride']:
    before = df[col].isna().sum()
    df[col] = clean_numeric_text(df[col])
    after = df[col].isna().sum()
    print(f"{col}: NaN before={before}, after={after} (equal => no numeric information lost)")

Loaded shape: (1048, 27)
Himoglobin: NaN before=51, after=51 (equal => no numeric information lost)
Potassium: NaN before=73, after=73 (equal => no numeric information lost)
Chloride: NaN before=78, after=78 (equal => no numeric information lost)


## 2. Troponin-I: censoring-aware parse + assay harmonisation

Two assay types on incompatible scales (`ng/mL` vs `ng/L`) are mixed in one
column. Converting High-Sensitivity (ng/L) values to ng/mL (÷1000) and
comparing per-assay ranges checks whether this harmonisation is sound.

In [3]:
def parse_troponin(v):
    if pd.isna(v):
        return np.nan, 0
    s = str(v).strip()
    if s.startswith('>') or s.startswith('<'):
        return float(s[1:]), 1
    try:
        return float(s), 0
    except ValueError:
        return np.nan, 0

parsed = df['Troponin-I'].apply(parse_troponin)
df['troponin_raw'] = parsed.apply(lambda t: t[0])
df['Troponin_Censored'] = parsed.apply(lambda t: t[1])

assay_col = 'Troponin- I assay type'
is_hs = df[assay_col] == 'High-Sensitivity Troponin-I (ng/L)'
df['Troponin_I_harmonised'] = df['troponin_raw']
df.loc[is_hs, 'Troponin_I_harmonised'] = df.loc[is_hs, 'troponin_raw'] / 1000.0

print("Before harmonisation:")
print(df.groupby(assay_col)['troponin_raw'].agg(['count', 'min', 'median', 'max']))
print("\nAfter harmonisation to ng/mL:")
print(df.groupby(assay_col)['Troponin_I_harmonised'].agg(['count', 'min', 'median', 'max']))
print("\nMedians converge to the same order of magnitude (~1.1-1.3), confirming the unit conversion is sound.")

Before harmonisation:
                                    count   min   median       max
Troponin- I assay type                                            
High-Sensitivity Troponin-I (ng/L)    202  2.50  1256.50  65860.00
Quantitative Troponin-I (ng/mL)       800  0.01     1.14     33.95

After harmonisation to ng/mL:
                                    count     min  median    max
Troponin- I assay type                                          
High-Sensitivity Troponin-I (ng/L)    202  0.0025  1.2565  65.86
Quantitative Troponin-I (ng/mL)       800  0.0100  1.1400  33.95

Medians converge to the same order of magnitude (~1.1-1.3), confirming the unit conversion is sound.


## 3. Population definition: exclude pediatric records

Per `dataset_audit.md`, all 13 records with Age<18 are `Heart Disease=1` —
retaining them would make "is this patient a child" a mechanical predictor.
This project's stated audience is adult risk screening, so excluding them is
appropriate for the defined research population (verified below, not just
assumed).

In [4]:
pediatric_mask = df['Age'] < 18
print(f"Excluding {pediatric_mask.sum()} pediatric records of {len(df)} total.")
print("Their Heart Disease distribution:", df.loc[pediatric_mask, 'Heart Disease'].value_counts().to_dict())
df_adult = df.loc[~pediatric_mask].reset_index(drop=True)
print("Adult population:", df_adult.shape, "| target balance:")
print(df_adult['Heart Disease'].value_counts(normalize=True).round(4))

Excluding 13 pediatric records of 1048 total.
Their Heart Disease distribution: {1: 13}
Adult population: (1035, 30) | target balance:
Heart Disease
1    0.5652
0    0.4348
Name: proportion, dtype: float64


## 4. Stratified 80/20 split — the 20% is not touched again until final evaluation

In [5]:
train_df, test_df = train_test_split(df_adult, test_size=0.20, stratify=df_adult[TARGET], random_state=RANDOM_STATE)
print("Train:", train_df.shape, "Test:", test_df.shape)
print("Train balance:\n", train_df[TARGET].value_counts(normalize=True).round(4))
print("Test balance:\n", test_df[TARGET].value_counts(normalize=True).round(4))
train_df.to_pickle("train.pkl")
test_df.to_pickle("test.pkl")

Train: (828, 30) Test: (207, 30)
Train balance:
 Heart Disease
1    0.5652
0    0.4348
Name: proportion, dtype: float64
Test balance:
 Heart Disease
1    0.5652
0    0.4348
Name: proportion, dtype: float64


## 5. Height / Weight / BMI representation experiment

BMI is a deterministic function of height and weight
(`weight_kg / height_m^2`). The user asked not to assume all three should be
independent model features — evaluated empirically here with a 5-fold CV
comparison of three representations, using two fast probe algorithms, on the
Basic feature set (train fold only).

In [6]:
def make_pipeline(numeric_cols, binary_cols, categorical_cols, model):
    numeric_pipe = Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)), ('scale', StandardScaler())])
    binary_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent'))])
    cat_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))])
    pre = ColumnTransformer([('num', numeric_pipe, numeric_cols), ('bin', binary_pipe, binary_cols), ('cat', cat_pipe, categorical_cols)])
    return Pipeline([('preprocess', pre), ('model', model)])

BASIC_BINARY = ['Family H/O', 'Hypertension', 'Diabetes', 'H/O ChestPain']
BASIC_CATEGORICAL = ['Sex']
variants = {
    'height_weight_only': ['Age', 'Height (cm)', 'Weight (kg)', 'BP(mmHg)'],
    'bmi_only':            ['Age', 'BMI', 'BP(mmHg)'],
    'all_three':           ['Age', 'Height (cm)', 'Weight (kg)', 'BMI', 'BP(mmHg)'],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, numeric_cols in variants.items():
    for model_name, model in [
        ('LogisticRegression', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
        ('RandomForest', RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, class_weight='balanced')),
    ]:
        pipe = make_pipeline(numeric_cols, BASIC_BINARY, BASIC_CATEGORICAL, model)
        scores = cross_val_score(pipe, train_df, train_df[TARGET], cv=cv, scoring='roc_auc')
        rows.append({'variant': name, 'model': model_name, 'mean_auc': scores.mean(), 'sd_auc': scores.std()})

result = pd.DataFrame(rows).pivot(index='variant', columns='model', values='mean_auc')
result

model,LogisticRegression,RandomForest
variant,,
all_three,0.904335,0.929008
bmi_only,0.898863,0.916973
height_weight_only,0.903957,0.925548


**Decision: use Height + Weight as model features; compute BMI for
display only, not as a separate model input.**

`all_three` beats `height_weight_only` by a negligible margin (well within
the ~0.025 CV standard deviation), while `bmi_only` is the consistent
underperformer of the three for both algorithms — reducing to BMI alone
loses a small amount of real signal that raw height and weight carry
separately. Adding BMI on top of height+weight adds no measurable value
since it's fully determined by them. This also satisfies the separate
frontend request to not ask the user to type a BMI value at all.

## 6. Final feature-set definitions used by the next notebook

In [7]:
BASIC_NUMERIC = ['Age', 'Height (cm)', 'Weight (kg)', 'BP(mmHg)']
BASIC_BINARY = ['Family H/O', 'Hypertension', 'Diabetes', 'H/O ChestPain']
BASIC_CATEGORICAL = ['Sex']
ENHANCED_NUMERIC_ADD = ['Total_Cholesterol(mg/dL)', 'HDL(mg/dL)', 'LDL(mg/dL)', 'Triglycerides(mg/dL)', 'RBS(mmol/L)', 'MaxHR']
ADVANCED_NUMERIC_ADD = ['Troponin_I_harmonised', 'Sodium(mmol/L)', 'Potassium', 'Chloride', 'Creatinine(mg/dL)', 'Platelets', 'Himoglobin']
ADVANCED_BINARY_ADD = ['Troponin_Censored']

print("A_Basic:            ", BASIC_NUMERIC + BASIC_BINARY + BASIC_CATEGORICAL)
print("B_Basic_Enhanced add:", ENHANCED_NUMERIC_ADD)
print("C_Advanced add:      ", ADVANCED_NUMERIC_ADD + ADVANCED_BINARY_ADD)
print("Excluded (never used as features): SL, UNIT, Troponin- I assay type (used only internally for harmonisation above)")

A_Basic:             ['Age', 'Height (cm)', 'Weight (kg)', 'BP(mmHg)', 'Family H/O', 'Hypertension', 'Diabetes', 'H/O ChestPain', 'Sex']
B_Basic_Enhanced add: ['Total_Cholesterol(mg/dL)', 'HDL(mg/dL)', 'LDL(mg/dL)', 'Triglycerides(mg/dL)', 'RBS(mmol/L)', 'MaxHR']
C_Advanced add:       ['Troponin_I_harmonised', 'Sodium(mmol/L)', 'Potassium', 'Chloride', 'Creatinine(mg/dL)', 'Platelets', 'Himoglobin', 'Troponin_Censored']
Excluded (never used as features): SL, UNIT, Troponin- I assay type (used only internally for harmonisation above)
